# <center> VAI Store - Engenharia de Variáveis </center>

---

## Conteúdo

1. [Imports](#imports)
1. [Dataset](#dataset)

<a id='imports'></a>
## 1. Imports

In [22]:
import pandas as pd

In [23]:
# Caminho dos módulos
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src' / 'utils'))

<a id='dataset'></a>
## 2. Datasets

In [24]:
df_vendas = pd.read_parquet('../data/treated/vendas_tratado.parquet')

# Leitura dos dados
df_vendas.info()
df_vendas.head()

<class 'pandas.core.frame.DataFrame'>
Index: 556198 entries, 000304564.299.0101042.20240102 to 000503479.257.0101032.20241231
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   COD_VENDA    556198 non-null  object        
 1   COD_FILIAL   556198 non-null  object        
 2   FILIAL       556198 non-null  object        
 3   DATA_ATEND   556198 non-null  datetime64[ns]
 4   SKU          556198 non-null  object        
 5   UNID         556198 non-null  object        
 6   QTD_VENDA    556198 non-null  float64       
 7   FATUR_VENDA  556198 non-null  float64       
 8   CLI_CPF      556198 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(6)
memory usage: 42.4+ MB


,COD_VENDA,COD_FILIAL,FILIAL,DATA_ATEND,SKU,UNID,QTD_VENDA,FATUR_VENDA,CLI_CPF
COD_ATEND,,,,,,,,,
000304564.299.0101042.20240102,000304564.299.0101042.20240102.000009.02,101042,SHOPPING,2024-01-02,9,KG,0.258,18.04,ec0abf3f4220
000374698.258.0101032.20240102,000374698.258.0101032.20240102.000009.10,101032,RUA,2024-01-02,9,KG,0.064,4.50,83d596ee0acf
000383551.256.0101032.20240102,000383551.256.0101032.20240102.000009.02,101032,RUA,2024-01-02,9,KG,0.164,11.49,46f63c18e055
000383533.256.0101032.20240102,000383533.256.0101032.20240102.000009.02,101032,RUA,2024-01-02,9,KG,0.202,14.13,6def1d35e2fe
000432095.257.0101032.20240102,000432095.257.0101032.20240102.000051.06,101032,RUA,2024-01-02,51,KG,0.086,6.87,1576e1a36394


In [25]:
df_produtos = pd.read_parquet('../data/treated/produtos_tratado.parquet')

# Leitura dos dados
df_produtos.info()
df_produtos.head()

<class 'pandas.core.frame.DataFrame'>
Index: 6499 entries, 3 to 171887
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   NOME_PRODUTO  6499 non-null   object
 1   CATEGORIA     6499 non-null   object
 2   SUBCATEGORIA  6499 non-null   object
dtypes: object(3)
memory usage: 203.1+ KB


,NOME_PRODUTO,CATEGORIA,SUBCATEGORIA
SKU,,,
3,COCO RALADO GROSSO KG,Doceria,Confeitaria
4,ICE TEA LEAO LATA 340ML,Sem categoria,Sem subcategoria
5,TAHINE ISTAMBUL 200G,Pelo Mundo,Pastas Árabes
6,AMENDOIM MOIDO KG,Castanhas & Oleaginosas,Oleaginosas moídas
7,HALAWI ISTAMBUL LATA 500G,Pelo Mundo,Pastas Árabes


### 2.1. Mesclando datasets

In [26]:
from merge_datasets import merge_datasets

df = merge_datasets(df_vendas, df_produtos, 'SKU')

## 3. Engenharia de Variáveis

### 3.1. Agregação dos datasets

In [27]:
df = df.groupby(['SKU', 'DATA_ATEND', 'FILIAL']).agg(
    QTD_TOTAL=('QTD_VENDA', 'sum'),
    FATUR_TOTAL=('FATUR_VENDA', 'sum'),
    CLIENTES_UNICOS=('CLI_CPF', 'nunique'),
).reset_index()

### 3.2. Extração de dados temporais

In [ ]:
df['DIA'] = df['DATA_ATEND'].dt.day
df['MES'] = df['DATA_ATEND'].dt.month
df['ANO'] = df['DATA_ATEND'].dt.year
df['DIA_SEMANA'] = df['DATA_ATEND'].dt.dayofweek

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30332 entries, 0 to 30331
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   SKU              30332 non-null  object        
 1   DATA_ATEND       30332 non-null  datetime64[ns]
 2   FILIAL           30332 non-null  object        
 3   QTD_TOTAL        30332 non-null  float64       
 4   FATUR_TOTAL      30332 non-null  float64       
 5   CLIENTES_UNICOS  30332 non-null  int64         
 6   DIA              30332 non-null  int32         
 7   MES              30332 non-null  int32         
 8   ANO              30332 non-null  int32         
 9   DIA_SEMANA       30332 non-null  int32         
dtypes: datetime64[ns](1), float64(2), int32(4), int64(1), object(2)
memory usage: 1.9+ MB


### 3.3. Criação de flags e índices sazonais

In [29]:
def vespera_to_bool(data):
    if data['DIA'] == 24 and data['MES'] == 12:
        return 1
    else:
        return 0

df['NATAL'] = df.apply(vespera_to_bool, axis=1)

In [30]:
def dias_ate_natal(data):
    ano = data.year
    data_natal = pd.to_datetime(f'{ano}-12-25')

    if data > data_natal:
        data_natal = pd.to_datetime(f'{ano + 1}-12-25')
    
    diff = data_natal - data
    return diff.days

df['DIAS_ATE_NATAL'] = df['DATA_ATEND'].apply(dias_ate_natal)